In [2]:
pip install pandas openpyxl scikit-learn

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import  LabelEncoder, StandardScaler
from sklearn.metrics import  accuracy_score, classification_report, confusion_matrix
from google.colab import drive
from pathlib import Path
import os
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown

#mount Gdrive
drive.mount("/content/drive")
base_path = Path('/content/drive/MyDrive/GenerativeAI')
print("Project files:", os.listdir(base_path))

file_path = base_path / "LoanApproval.xlsx"

#Read file data
df = pd.read_excel(file_path)


#convert string column values into int
label_encoder = LabelEncoder()
df["EmploymentType"] = label_encoder.fit_transform(df["EmploymentType"])


x = df[["Age","AnnualIncome(lakhs)","CreditScore(300-900)","LoanAmount(lakhs)","LoanTerm(years)","EmploymentType"]]
y = df["loan(yes/no)"]

#split dataset
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

#
# 6. Scale features (important for KNN)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)
#print(f"\n x_train scaled {x_train_scaled}")

#create and train KNN model with scaled data
#n_neighbors is the value for number of nearest record as per given data . based on those record answer majority the predcition will find
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(x_train_scaled, y_train)


#In Decision tree we don't need the scalling.
decisionTree_model = DecisionTreeClassifier(random_state = 45)
decisionTree_model.fit(x_train, y_train)

#Prediction the answer for the test model
y_prediction = knn_model.predict(x_test_scaled)

dt_y_prediction = decisionTree_model.predict(x_test)

print(f"\n data for testing without scalling")
for index, row in x_test.iterrows():
    print(f"\nRow {index}")
    for column in x_test.columns:
        print(f"{column}: {row[column]}")

print(f"\n Data for prediction {x_test_scaled}")
print(f"\n Prediction. value is using KNN  {y_prediction}")
print(f"\n Prediction. value is using Decision Tree  {dt_y_prediction}")

#Evalute the model

#Accuracy: Accuracy measures how many predictions were correct., Correct = 3 , Total = 4 : Accuracy = 3 / 4 = 0.75 (75%)
accuracy = accuracy_score(y_test, y_prediction)
dt_accuracy = accuracy_score(y_test, dt_y_prediction)

#Confusion matrix shows how predictions are distributed.: 
cm = confusion_matrix(y_test, y_prediction)
dt_cm = confusion_matrix(y_test, dt_y_prediction)
#This provides detailed metrics:
#Precision	how many predicted positives are correct
#Recall	how many actual positives were detected
#F1-score	balance of precision and recall
#Support	number of samples in each class 

report = classification_report(y_test, y_prediction)
dt_report = classification_report(y_test, dt_y_prediction)

print(f"\n Accuracy of model using KNN{accuracy}")
print(f"\n Accuracy of model using Decision tree {dt_accuracy}")

print(f"\n confussion matrix of model using KNN {cm}")
print(f"\n confussion matrix of model using Decision tree {dt_cm}")

print(f"\n report for model using KNN {report}")
print(f"\n report for model using Decision tre {dt_report}")

#compare the actual vs predicted
result = pd.DataFrame({"Default loan value ": y_test.values, "prediction loadn value for KNN": y_prediction, "prediction loadn value for Decision tree": dt_y_prediction})

print(f"\n Actual vs prediction valye {result}")

plot_df = x_test.copy().reset_index(drop=True)
y_test_plot = pd.Series(y_test).reset_index(drop=True)
knn_pred_plot = pd.Series(y_prediction).reset_index(drop=True)
dt_pred_plot = pd.Series(dt_y_prediction).reset_index(drop=True)

# Add actual and predicted values into one dataframe
plot_df["Actual"] = y_test_plot
plot_df["KNN_Prediction"] = knn_pred_plot
plot_df["DT_Prediction"] = dt_pred_plot

feature_columns = list(x_test_plot.columns)

def plot_comparison(x_feature, y_feature):
    plt.figure(figsize=(7, 5))

    # 1. Actual points
    actual_0 = plot_df[plot_df["Actual"] == 0]
    actual_1 = plot_df[plot_df["Actual"] == 1]

    plt.scatter(
        actual_0[x_feature],
        actual_0[y_feature],
        label="Actual: No Default (0)",
        s=180,
        marker="o",
        alpha=0.6
    )

    plt.scatter(
        actual_1[x_feature],
        actual_1[y_feature],
        label="Actual: Default (1)",
        s=180,
        marker="o",
        alpha=0.6
    )

    # 2. KNN prediction markers
    knn_0 = plot_df[plot_df["KNN_Prediction"] == 0]
    knn_1 = plot_df[plot_df["KNN_Prediction"] == 1]

    plt.scatter(
        knn_0[x_feature],
        knn_0[y_feature],
        label="KNN Pred: 0",
        s=90,
        marker="s"
    )

    plt.scatter(
        knn_1[x_feature],
        knn_1[y_feature],
        label="KNN Pred: 1",
        s=90,
        marker="s"
    )

    # 3. Decision Tree prediction markers
    dt_0 = plot_df[plot_df["DT_Prediction"] == 0]
    dt_1 = plot_df[plot_df["DT_Prediction"] == 1]

    plt.scatter(
        dt_0[x_feature],
        dt_0[y_feature],
        label="DT Pred: 0",
        s=110,
        marker="^"
    )

    plt.scatter(
        dt_1[x_feature],
        dt_1[y_feature],
        label="DT Pred: 1",
        s=110,
        marker="^"
    )

    plt.xlabel(x_feature)
    plt.ylabel(y_feature)
    plt.title(f"Actual vs KNN vs Decision Tree\n({x_feature} vs {y_feature})")
    plt.legend()
    plt.grid(True)
    plt.show()
# Interactive dropdown
interact(
    plot_comparison,
    x_feature=Dropdown(options=feature_columns, value=feature_columns[0], description="X-axis"),
    y_feature=Dropdown(options=feature_columns, value=feature_columns[1], description="Y-axis")
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project files: ['EmployeProductivity.xlsx', 'LoanApproval.xlsx']

 data for testing without scalling

Row 8
Age: 38.0
AnnualIncome(lakhs): 9.0
CreditScore(300-900): 700.0
LoanAmount(lakhs): 7.0
LoanTerm(years): 8.0
EmploymentType: 0.0

Row 1
Age: 45.0
AnnualIncome(lakhs): 12.0
CreditScore(300-900): 680.0
LoanAmount(lakhs): 10.0
LoanTerm(years): 10.0
EmploymentType: 1.0

 Data for prediction [[-0.12101891 -0.3200922   0.33333333 -0.33752637 -0.26620695 -0.77459667]
 [ 0.55668699  0.44812908 -0.11111111  0.56254395  0.20704985  1.29099445]]

 Prediction. value is using KNN  [0 1]

 Prediction. value is using Decision Tree  [0 1]

 Accuracy of model using KNN1.0

 Accuracy of model using Decision tree 1.0

 confussion matrix of model using KNN [[1 0]
 [0 1]]

 confussion matrix of model using Decision tree [[1 0]
 [0 1]]

 report for model using KNN             

interactive(children=(Dropdown(description='X-axis', options=('Age', 'AnnualIncome(lakhs)', 'CreditScore(300-9…

<function __main__.plot_comparison(x_feature, y_feature)>